# Studi Komparatif U-Net dan Attention U-Net untuk Segmentasi Lesi Kulit (ISIC 2018)
---

## List of contents
1. Introduction  
2. EDA (Exploratory Data Analysis)  
3. Dasar Teori
4. Pipeline   
5. Implementasi & Training  
6. Evaluasi & Perbandingan Model  
7. Kesimpulan dan Saran  
8. Referensi


**Dataset yang digunakan:** ISIC 2018 Challenge – *Lesion Boundary Segmentation (Task 1)*  
Target: membandingkan performa **U-Net** vs **Attention U-Net** untuk segmentasi mask biner lesi kulit.


## **1. Pendahuluan**

### Latar Belakang
Segmentasi lesi kulit dari citra dermoskopi penting untuk membantu analisis klinis (misalnya menaksir batas lesi untuk diagnosis lebih lanjut).  
Model CNN arsitektur encoder–decoder seperti **U-Net** menjadi baseline populer.  
Pengembangan seperti **Attention U-Net** menambahkan *attention gate* agar model fokus pada area penting (lesi) dan menekan fitur yang tidak relevan.

### Tujuan
Membandingkan U-Net dan Attention U-Net pada dataset ISIC 2018 Task 1 menggunakan metrik:
- **IoU (Jaccard Index)**
- **Dice Coefficient**

### Rumusan Masalah
Apakah Attention U-Net memberikan peningkatan performa segmentasi dibanding U-Net standar pada ISIC 2018?


## **Setup & Konfigurasi**
Pada bagian ini:
- Import library
- Atur random seed (reproducible)
- Menentukan path dataset dan hyperparameter


In [5]:
from pathlib import Path
import random, math

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm.auto import tqdm

def seed_everything(seed: int = 42, deterministic: bool = False):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    torch.backends.cudnn.deterministic = deterministic
    torch.backends.cudnn.benchmark = not deterministic

seed_everything(42, deterministic=False)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

device(type='cuda')

In [6]:
CFG = {
    "DATA_ROOT": Path("data/isic2018_raw"),
    "IMAGES_DIR": "ISIC2018_Task1-2_Training_Input",
    "MASKS_DIR": "ISIC2018_Task1_Training_GroundTruth",

    "IMG_SIZE": 256,
    "BATCH_SIZE": 8,
    "NUM_WORKERS": 2,

    "EPOCHS": 10,
    "LR": 1e-3,
    "WEIGHT_DECAY": 1e-5,

    "VAL_RATIO": 0.1,
    "TEST_RATIO": 0.1,

    "THRESHOLD": 0.5,
    "SAVE_DIR": Path("checkpoints"),
}

CFG["SAVE_DIR"].mkdir(parents=True, exist_ok=True)

images_path = CFG["DATA_ROOT"] / CFG["IMAGES_DIR"]
masks_path  = CFG["DATA_ROOT"] / CFG["MASKS_DIR"]

images_path, masks_path

(PosixPath('data/isic2018_raw/ISIC2018_Task1-2_Training_Input'),
 PosixPath('data/isic2018_raw/ISIC2018_Task1_Training_GroundTruth'))

## **Download & Extract ISIC 2018**

In [4]:
!pip -q install pooch

import pooch, zipfile
from pathlib import Path

URLS = {
  "train_input": "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1-2_Training_Input.zip",
  "train_gt":    "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1_Training_GroundTruth.zip",
  "val_input":   "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1-2_Validation_Input.zip",
  "test_input":  "https://isic-archive.s3.amazonaws.com/challenges/2018/ISIC2018_Task1-2_Test_Input.zip",
}

RAW_DIR = Path(CFG["DATA_ROOT"])
RAW_DIR.mkdir(parents=True, exist_ok=True)

for name, url in URLS.items():
    zip_path = Path(pooch.retrieve(url=url, fname=Path(url).name, path=RAW_DIR, known_hash=None))
    out_dir = RAW_DIR / zip_path.stem
    if out_dir.exists() and any(out_dir.iterdir()):
        print(f"[SKIP] {out_dir}")
        continue
    with zipfile.ZipFile(zip_path) as z:
        z.extractall(RAW_DIR)
    print(f"[READY] {out_dir}")


SHA256 hash of downloaded file: 80f98572347a2d7a376227fa9eb2e4f7459d317cb619865b8b9910c81446675f
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.


[READY] data/isic2018_raw/ISIC2018_Task1-2_Training_Input


SHA256 hash of downloaded file: 99f8b2bb3c4d6af483362010715f7e7d5d122d9f6c02cac0e0d15bef77c7604c
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.


[READY] data/isic2018_raw/ISIC2018_Task1_Training_GroundTruth


SHA256 hash of downloaded file: 0ea920fcfe512d12a6e620b50b50233c059f67b10146e1479c82be58ff15a797
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.


[READY] data/isic2018_raw/ISIC2018_Task1-2_Validation_Input


SHA256 hash of downloaded file: e59ae1f69f4ed16f09db2cb1d76c2a828487b63d28f6ab85997f5616869b127d
Use this value as the 'known_hash' argument of 'pooch.retrieve' to ensure that the file hasn't changed if it is downloaded again in the future.


[READY] data/isic2018_raw/ISIC2018_Task1-2_Test_Input


## **2. EDA**

EDA yang dilakukan:
- Cek jumlah data dan kesesuaian pasangan image–mask
- Visualisasi contoh image + overlay mask
- Distribusi ukuran gambar (H×W)
- Distribusi luas lesi (persentase area mask)


In [7]:
def list_pairs(images_dir: Path, masks_dir: Path, ext="jpg"):
    imgs = sorted(images_dir.glob(f"*.{ext}"))
    pairs = [(p, masks_dir / f"{p.stem}.png") for p in imgs if (masks_dir / f"{p.stem}.png").exists()]
    return pairs, len(imgs) - len(pairs)

pairs, missing = list_pairs(images_path, masks_path, ext="jpg")
print(f"[pairs] {len(pairs)} [missing_masks] {missing}")

NameError: name 'list_image_mask_pairs' is not defined

In [ ]:
def read_rgb(p):  return np.array(Image.open(p).convert("RGB"))
def read_mask(p): return (np.array(Image.open(p).convert("L")) > 0).astype(np.uint8)

def show_samples(pairs, n=9, seed=42, alpha=0.35):
    rnd = random.Random(seed)
    sample = rnd.sample(pairs, k=min(n, len(pairs)))

    cols = 3
    rows = math.ceil(len(sample) / cols)
    plt.figure(figsize=(cols*5, rows*5))

    for i, (img_p, mask_p) in enumerate(sample, 1):
        img = read_rgb(img_p)
        m = read_mask(mask_p).astype(bool)

        out = img.copy()
        out[m] = ((1 - alpha) * out[m] + alpha * np.array([255, 0, 0])).astype(np.uint8)

        plt.subplot(rows, cols, i)
        plt.imshow(out)
        plt.title(img_p.name)
        plt.axis("off")

    plt.tight_layout()
    plt.show()

if pairs:
    show_samples(pairs, n=9, seed=42)

In [ ]:
# Distribusi ukuran gambar
def compute_image_sizes(pairs, max_n=500):
    sample = pairs[:min(max_n, len(pairs))]
    sizes = []
    for img_p, _ in tqdm(sample, desc="Reading sizes"):
        with Image.open(img_p) as im:
            w, h = im.size
        sizes.append((h, w))
    return pd.DataFrame(sizes, columns=["H", "W"])

if len(pairs) > 0:
    df_sizes = compute_image_sizes(pairs, max_n=500)
    display(df_sizes.describe())

    plt.figure(figsize=(6,4))
    plt.hist(df_sizes["H"], bins=30)
    plt.title("Distribusi Height (sample)")
    plt.xlabel("Height")
    plt.ylabel("Count")
    plt.show()

    plt.figure(figsize=(6,4))
    plt.hist(df_sizes["W"], bins=30)
    plt.title("Distribusi Width (sample)")
    plt.xlabel("Width")
    plt.ylabel("Count")
    plt.show()


In [ ]:
# Distribusi luas lesi (persentase area mask)
def compute_lesion_area_ratio(pairs, max_n=500):
    sample = pairs[:min(max_n, len(pairs))]
    ratios = []
    for _, mask_p in tqdm(sample, desc="Reading masks"):
        m = (np.array(Image.open(mask_p).convert("L")) > 0).astype(np.uint8)
        m = (m > 0).astype(np.uint8)
        ratios.append(float(m.mean()))  # proporsi pixel lesi
    return pd.DataFrame({"lesion_area_ratio": ratios})

if len(pairs) > 0:
    df_area = compute_lesion_area_ratio(pairs, max_n=500)
    display(df_area.describe())

    plt.figure(figsize=(6,4))
    plt.hist(df_area["lesion_area_ratio"], bins=30)
    plt.title("Distribusi Luas Lesi (proporsi pixel mask=1)")
    plt.xlabel("Lesion area ratio")
    plt.ylabel("Count")
    plt.show()


## **3. Dasar Teori**

### 3.1 U-Net (Ronneberger et al.)
U-Net adalah arsitektur encoder–decoder dengan **skip connections**:
- **Encoder** mengekstrak fitur semantik (downsampling).
- **Decoder** mengembalikan resolusi spasial (upsampling).
- Skip connection menggabungkan fitur resolusi tinggi dari encoder agar detail batas objek terjaga.

### 3.2 Attention U-Net (Oktay et al.)
Attention U-Net menambahkan **Attention Gate (AG)** di jalur skip connection:
- AG menerima fitur dari encoder (x) dan sinyal gating dari decoder (g).
- AG menghasilkan koefisien attention untuk menekankan area relevan (lesi) dan menekan noise/background.

> Dalam notebook ini, kedua model dilatih dengan pipeline yang sama agar perbandingan adil.


## **4. Pipeline**
1. **Load data** (image, mask)  
2. **Preprocessing**: resize, normalisasi [0,1]  
3. **Split data**: train/val/test  
4. **Training**:  
   - Loss = BCEWithLogits + Dice Loss  
   - Optimizer = Adam  
5. **Evaluasi** pada test set: IoU & Dice  
6. **Analisis kualitatif**: tampilkan prediksi mask dari kedua model


## **5. Implementasi**
Bagian ini berisi:
- Dataset class & DataLoader
- Implementasi U-Net
- Implementasi Attention U-Net
- Loss & metric
- Training loop & evaluasi


In [ ]:
# =========================
# Dataset & DataLoader
# =========================

def resize_pil(im: Image.Image, size: int, is_mask: bool = False):
    if is_mask:
        return im.resize((size, size), resample=Image.NEAREST)
    return im.resize((size, size), resample=Image.BILINEAR)

class ISICSegDataset(Dataset):
    def __init__(self, pairs, img_size=256, augment=False):
        self.pairs = pairs
        self.img_size = img_size
        self.augment = augment

    def __len__(self):
        return len(self.pairs)

    def _simple_augment(self, img, mask):
        # augment sederhana tanpa dependency tambahan
        if random.random() < 0.5:
            img = np.flip(img, axis=1).copy()
            mask = np.flip(mask, axis=1).copy()
        if random.random() < 0.2:
            img = np.flip(img, axis=0).copy()
            mask = np.flip(mask, axis=0).copy()
        return img, mask

    def __getitem__(self, idx):
        img_path, mask_path = self.pairs[idx]
        img = read_rgb(img_path)
        mask = read_mask(mask_path)

        img = resize_pil(img, self.img_size, is_mask=False)
        mask = resize_pil(mask, self.img_size, is_mask=True)

        img = np.array(img).astype(np.float32) / 255.0  # HWC
        mask = (np.array(mask) > 0).astype(np.float32)  # HW

        if self.augment:
            img, mask = self._simple_augment(img, mask)

        img_t = torch.from_numpy(img).permute(2, 0, 1)   # 3,H,W
        mask_t = torch.from_numpy(mask).unsqueeze(0)     # 1,H,W
        return img_t, mask_t

def make_splits(pairs, val_ratio=0.1, test_ratio=0.1, seed=42):
    idx = np.arange(len(pairs))
    rng = np.random.default_rng(seed)
    rng.shuffle(idx)

    n = len(idx)
    n_test = int(n * test_ratio)
    n_val  = int(n * val_ratio)

    test_idx = idx[:n_test]
    val_idx  = idx[n_test:n_test+n_val]
    train_idx= idx[n_test+n_val:]

    pick = lambda ii: [pairs[i] for i in ii]
    return pick(train_idx), pick(val_idx), pick(test_idx)

if len(pairs) > 0:
    train_pairs, val_pairs, test_pairs = make_splits(pairs, CFG["VAL_RATIO"], CFG["TEST_RATIO"], seed=42)
    print("train/val/test:", len(train_pairs), len(val_pairs), len(test_pairs))


In [ ]:
if len(pairs) > 0:
    train_ds = ISICSegDataset(train_pairs, img_size=CFG["IMG_SIZE"], augment=True)
    val_ds   = ISICSegDataset(val_pairs,   img_size=CFG["IMG_SIZE"], augment=False)
    test_ds  = ISICSegDataset(test_pairs,  img_size=CFG["IMG_SIZE"], augment=False)

    train_loader = DataLoader(train_ds, batch_size=CFG["BATCH_SIZE"], shuffle=True,
                              num_workers=CFG["NUM_WORKERS"], pin_memory=True)
    val_loader   = DataLoader(val_ds, batch_size=CFG["BATCH_SIZE"], shuffle=False,
                              num_workers=CFG["NUM_WORKERS"], pin_memory=True)
    test_loader  = DataLoader(test_ds, batch_size=CFG["BATCH_SIZE"], shuffle=False,
                              num_workers=CFG["NUM_WORKERS"], pin_memory=True)


In [ ]:
# =========================
# Building blocks
# =========================

class DoubleConv(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.Conv2d(in_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),

            nn.Conv2d(out_ch, out_ch, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_ch),
            nn.ReLU(inplace=True),
        )
    def forward(self, x):
        return self.net(x)

class Down(nn.Module):
    def __init__(self, in_ch, out_ch):
        super().__init__()
        self.net = nn.Sequential(
            nn.MaxPool2d(2),
            DoubleConv(in_ch, out_ch)
        )
    def forward(self, x):
        return self.net(x)

class Up(nn.Module):
    def __init__(self, in_ch, out_ch, bilinear=True):
        super().__init__()
        self.bilinear = bilinear
        if bilinear:
            self.up = nn.Upsample(scale_factor=2, mode="bilinear", align_corners=True)
            self.conv = DoubleConv(in_ch, out_ch)
        else:
            self.up = nn.ConvTranspose2d(in_ch // 2, in_ch // 2, kernel_size=2, stride=2)
            self.conv = DoubleConv(in_ch, out_ch)

    def forward(self, x1, x2):
        x1 = self.up(x1)
        diffY = x2.size()[2] - x1.size()[2]
        diffX = x2.size()[3] - x1.size()[3]
        x1 = F.pad(x1, [diffX//2, diffX-diffX//2, diffY//2, diffY-diffY//2])
        x = torch.cat([x2, x1], dim=1)
        return self.conv(x)

class OutConv(nn.Module):
    def __init__(self, in_ch, out_ch=1):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, kernel_size=1)
    def forward(self, x):
        return self.conv(x)


In [ ]:
# =========================
# U-Net
# =========================

class UNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base_ch=64, bilinear=True):
        super().__init__()
        self.inc = DoubleConv(in_ch, base_ch)
        self.down1 = Down(base_ch, base_ch*2)
        self.down2 = Down(base_ch*2, base_ch*4)
        self.down3 = Down(base_ch*4, base_ch*8)
        factor = 2 if bilinear else 1
        self.down4 = Down(base_ch*8, base_ch*16//factor)

        self.up1 = Up(base_ch*16, base_ch*8//factor, bilinear)
        self.up2 = Up(base_ch*8,  base_ch*4//factor, bilinear)
        self.up3 = Up(base_ch*4,  base_ch*2//factor, bilinear)
        self.up4 = Up(base_ch*2,  base_ch, bilinear)
        self.outc = OutConv(base_ch, out_ch)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        x = self.up1(x5, x4)
        x = self.up2(x,  x3)
        x = self.up3(x,  x2)
        x = self.up4(x,  x1)
        return self.outc(x)


In [ ]:
# =========================
# Attention Gate & Attention U-Net
# =========================

class AttentionGate(nn.Module):
    """Attention gate (AG): x = encoder feature, g = gating (decoder feature)."""
    def __init__(self, F_g, F_l, F_int):
        super().__init__()
        self.W_g = nn.Sequential(
            nn.Conv2d(F_g, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.W_x = nn.Sequential(
            nn.Conv2d(F_l, F_int, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(F_int)
        )
        self.psi = nn.Sequential(
            nn.Conv2d(F_int, 1, kernel_size=1, stride=1, padding=0, bias=True),
            nn.BatchNorm2d(1),
            nn.Sigmoid()
        )
        self.relu = nn.ReLU(inplace=True)

    def forward(self, x, g):
        g1 = self.W_g(g)
        x1 = self.W_x(x)
        psi = self.relu(g1 + x1)
        psi = self.psi(psi)
        return x * psi

class AttUNet(nn.Module):
    def __init__(self, in_ch=3, out_ch=1, base_ch=64, bilinear=True):
        super().__init__()
        self.inc = DoubleConv(in_ch, base_ch)
        self.down1 = Down(base_ch, base_ch*2)
        self.down2 = Down(base_ch*2, base_ch*4)
        self.down3 = Down(base_ch*4, base_ch*8)
        factor = 2 if bilinear else 1
        self.down4 = Down(base_ch*8, base_ch*16//factor)

        self.up1 = Up(base_ch*16, base_ch*8//factor, bilinear)
        self.up2 = Up(base_ch*8,  base_ch*4//factor, bilinear)
        self.up3 = Up(base_ch*4,  base_ch*2//factor, bilinear)
        self.up4 = Up(base_ch*2,  base_ch, bilinear)

        self.ag1 = AttentionGate(F_g=base_ch*16//factor, F_l=base_ch*8, F_int=base_ch*4)
        self.ag2 = AttentionGate(F_g=base_ch*8//factor,  F_l=base_ch*4, F_int=base_ch*2)
        self.ag3 = AttentionGate(F_g=base_ch*4//factor,  F_l=base_ch*2, F_int=base_ch)
        self.ag4 = AttentionGate(F_g=base_ch*2//factor,  F_l=base_ch,   F_int=max(1, base_ch//2))

        self.outc = OutConv(base_ch, out_ch)

    def forward(self, x):
        x1 = self.inc(x)
        x2 = self.down1(x1)
        x3 = self.down2(x2)
        x4 = self.down3(x3)
        x5 = self.down4(x4)

        # gating signals from decoder features (upsampled as needed)
        g4 = F.interpolate(x5, size=x4.shape[-2:], mode="bilinear", align_corners=True)
        x4_att = self.ag1(x4, g4)
        d1 = self.up1(x5, x4_att)

        g3 = F.interpolate(d1, size=x3.shape[-2:], mode="bilinear", align_corners=True)
        x3_att = self.ag2(x3, g3)
        d2 = self.up2(d1, x3_att)

        g2 = F.interpolate(d2, size=x2.shape[-2:], mode="bilinear", align_corners=True)
        x2_att = self.ag3(x2, g2)
        d3 = self.up3(d2, x2_att)

        g1 = F.interpolate(d3, size=x1.shape[-2:], mode="bilinear", align_corners=True)
        x1_att = self.ag4(x1, g1)
        d4 = self.up4(d3, x1_att)

        return self.outc(d4)


In [ ]:
# =========================
# Loss & Metrics
# =========================

def dice_coef(pred, target, eps=1e-7):
    pred = pred.contiguous()
    target = target.contiguous()
    intersection = (pred * target).sum(dim=(2,3))
    union = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
    dice = (2.0 * intersection + eps) / (union + eps)
    return dice.mean()

def iou_score(pred, target, eps=1e-7):
    pred = pred.contiguous()
    target = target.contiguous()
    intersection = (pred * target).sum(dim=(2,3))
    total = pred.sum(dim=(2,3)) + target.sum(dim=(2,3))
    union = total - intersection
    iou = (intersection + eps) / (union + eps)
    return iou.mean()

class DiceLoss(nn.Module):
    def __init__(self, eps=1e-7):
        super().__init__()
        self.eps = eps
    def forward(self, logits, targets):
        probs = torch.sigmoid(logits)
        return 1.0 - dice_coef(probs, targets, eps=self.eps)

bce_loss = nn.BCEWithLogitsLoss()
dice_loss = DiceLoss()

def combined_loss(logits, targets, alpha=0.5):
    return alpha * bce_loss(logits, targets) + (1 - alpha) * dice_loss(logits, targets)


In [ ]:
# =========================
# Training & Evaluation
# =========================

@torch.no_grad()
def evaluate(model, loader, threshold=0.5):
    model.eval()
    losses, dices, ious = [], [], []
    for imgs, masks in loader:
        imgs = imgs.to(device)
        masks = masks.to(device)

        logits = model(imgs)
        loss = combined_loss(logits, masks)

        probs = torch.sigmoid(logits)
        preds = (probs >= threshold).float()

        losses.append(loss.item())
        dices.append(dice_coef(preds, masks).item())
        ious.append(iou_score(preds, masks).item())

    return {
        "loss": float(np.mean(losses)) if losses else None,
        "dice": float(np.mean(dices)) if dices else None,
        "iou":  float(np.mean(ious)) if ious else None,
    }

def train_one_epoch(model, loader, optimizer, scaler=None, threshold=0.5):
    model.train()
    losses, dices, ious = [], [], []

    for imgs, masks in tqdm(loader, leave=False):
        imgs = imgs.to(device)
        masks = masks.to(device)

        optimizer.zero_grad(set_to_none=True)

        if scaler is not None:
            with torch.amp.autocast(device_type="cuda", enabled=use_amp):
                logits = model(imgs)
                loss = combined_loss(logits, masks)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            logits = model(imgs)
            loss = combined_loss(logits, masks)
            loss.backward()
            optimizer.step()

        probs = torch.sigmoid(logits).detach()
        preds = (probs >= threshold).float()

        losses.append(loss.item())
        dices.append(dice_coef(preds, masks).item())
        ious.append(iou_score(preds, masks).item())

    return {
        "loss": float(np.mean(losses)),
        "dice": float(np.mean(dices)),
        "iou":  float(np.mean(ious)),
    }

def fit(model, train_loader, val_loader, epochs=10, lr=1e-3, weight_decay=1e-5, model_name="model"):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=weight_decay)

    use_amp = torch.cuda.is_available()
    scaler = torch.amp.GradScaler("cuda") if use_amp else None

    history = []
    best_val_iou = -1.0

    for epoch in range(1, epochs+1):
        tr = train_one_epoch(model, train_loader, optimizer, scaler=scaler, threshold=CFG["THRESHOLD"])
        va = evaluate(model, val_loader, threshold=CFG["THRESHOLD"])

        row = {
            "epoch": epoch,
            **{f"train_{k}": v for k, v in tr.items()},
            **{f"val_{k}": v for k, v in va.items()},
        }
        history.append(row)

        print(
            f"[{model_name}] Epoch {epoch:02d}/{epochs} | "
            f"train loss {tr['loss']:.4f} dice {tr['dice']:.4f} iou {tr['iou']:.4f} | "
            f"val loss {va['loss']:.4f} dice {va['dice']:.4f} iou {va['iou']:.4f}"
        )

        if va["iou"] is not None and va["iou"] > best_val_iou:
            best_val_iou = va["iou"]
            ckpt_path = CFG["SAVE_DIR"] / f"{model_name}_best.pt"
            torch.save(model.state_dict(), ckpt_path)

    return model, pd.DataFrame(history)

def plot_history(df_hist, title_prefix=""):
    if df_hist is None or len(df_hist) == 0:
        return

    plt.figure(figsize=(6,4))
    plt.plot(df_hist["epoch"], df_hist["train_loss"], label="train_loss")
    plt.plot(df_hist["epoch"], df_hist["val_loss"], label="val_loss")
    plt.title(f"{title_prefix} Loss")
    plt.xlabel("Epoch")
    plt.ylabel("Loss")
    plt.legend()
    plt.show()

    plt.figure(figsize=(6,4))
    plt.plot(df_hist["epoch"], df_hist["train_iou"], label="train_iou")
    plt.plot(df_hist["epoch"], df_hist["val_iou"], label="val_iou")
    plt.title(f"{title_prefix} IoU")
    plt.xlabel("Epoch")
    plt.ylabel("IoU")
    plt.legend()
    plt.show()

    plt.figure(figsize=(6,4))
    plt.plot(df_hist["epoch"], df_hist["train_dice"], label="train_dice")
    plt.plot(df_hist["epoch"], df_hist["val_dice"], label="val_dice")
    plt.title(f"{title_prefix} Dice")
    plt.xlabel("Epoch")
    plt.ylabel("Dice")
    plt.legend()
    plt.show()


## **6. Training: U-Net vs Attention U-Net**

- training U-Net
- training Attention U-Net
- evaluasi pada test set
- menampilkan tabel hasil perbandingan


In [ ]:
def run_experiment(model_factory, model_name):
    if len(pairs) == 0:
        raise RuntimeError(
            "Dataset belum tersedia. Pastikan CFG['DATA_ROOT'] benar dan folder images/masks ada."
        )

    model = model_factory()
    model, hist = fit(
        model,
        train_loader=train_loader,
        val_loader=val_loader,
        epochs=CFG["EPOCHS"],
        lr=CFG["LR"],
        weight_decay=CFG["WEIGHT_DECAY"],
        model_name=model_name,
    )

    plot_history(hist, title_prefix=model_name)

    # load best checkpoint
    ckpt_path = CFG["SAVE_DIR"] / f"{model_name}_best.pt"
    if ckpt_path.exists():
        model.load_state_dict(torch.load(ckpt_path, map_location=device))

    test_metrics = evaluate(model, test_loader, threshold=CFG["THRESHOLD"])
    return model, hist, test_metrics

UNetFactory = lambda: UNet(in_ch=3, out_ch=1, base_ch=64, bilinear=True)
AttUNetFactory = lambda: AttUNet(in_ch=3, out_ch=1, base_ch=64, bilinear=True)


In [ ]:
# --- U-Net ---
unet_model, unet_hist, unet_test = run_experiment(UNetFactory, "unet")

# --- Attention U-Net ---
att_model, att_hist, att_test = run_experiment(AttUNetFactory, "att_unet")

In [ ]:
results = pd.DataFrame([
    {"model": "U-Net", **unet_test},
    {"model": "Attention U-Net", **att_test},
])
display(results)

plt.figure(figsize=(6,4))
plt.bar(results["model"], results["iou"])
plt.title("Perbandingan IoU (Test)")
plt.ylabel("IoU")
plt.show()

plt.figure(figsize=(6,4))
plt.bar(results["model"], results["dice"])
plt.title("Perbandingan Dice (Test)")
plt.ylabel("Dice")
plt.show()


## **7. Evaluasi Kualitatif**

Bagian ini menampilkan beberapa contoh:
- input image
- ground truth mask
- prediksi U-Net
- prediksi Attention U-Net


In [ ]:
@torch.no_grad()
def predict_mask(model, img_t):
    model.eval()
    img_t = img_t.unsqueeze(0).to(device)  # 1,3,H,W
    logits = model(img_t)
    prob = torch.sigmoid(logits)[0, 0].cpu().numpy()
    pred = (prob >= CFG["THRESHOLD"]).astype(np.uint8)
    return prob, pred

def visualize_predictions(unet_model, att_model, dataset, n=6, seed=123):
    rng = random.Random(seed)
    idxs = rng.sample(range(len(dataset)), k=min(n, len(dataset)))

    plt.figure(figsize=(14, 4*n))
    for i, idx in enumerate(idxs, 1):
        img_t, mask_t = dataset[idx]
        img = (img_t.permute(1,2,0).numpy() * 255).astype(np.uint8)
        gt  = mask_t[0].numpy().astype(np.uint8)

        _, unet_pred = predict_mask(unet_model, img_t)
        _, att_pred  = predict_mask(att_model, img_t)

        for j, (title, arr, cmap) in enumerate([
            ("Image", img, None),
            ("GT Mask", gt, "gray"),
            ("U-Net Pred", unet_pred, "gray"),
            ("Att U-Net Pred", att_pred, "gray"),
        ]):
            plt.subplot(n, 4, (i-1)*4 + (j+1))
            if arr.ndim == 3:
                plt.imshow(arr)
            else:
                plt.imshow(arr, cmap=cmap)
            plt.title(title)
            plt.axis("off")

    plt.tight_layout()
    plt.show()

visualize_predictions(unet_model, att_model, test_ds, n=6, seed=123)


## Kesimpulan dan Saran

### Kesimpulan (isi setelah hasil keluar)
Tuliskan ringkasan berdasarkan tabel metrik:
- Model mana yang unggul pada IoU dan Dice
- Apakah perbedaannya terlihat juga secara visual

### Saran Pengembangan
1. **Augmentasi lebih kaya** (rotasi kecil, brightness/contrast, random crop) menggunakan `albumentations`
2. **Loss alternatif**: Tversky / Focal Tversky untuk menangani lesi kecil
3. **Cross-validation** (mis. 5-fold) untuk evaluasi lebih robust
4. **Input size 512** untuk detail batas lebih halus (resource lebih berat)
5. Laporkan **parameter count** dan **inference time** untuk trade-off performa vs kompleksitas


## Referensi

Codella, N., Rotemberg, V., Tschandl, P., Celebi, M. E., Dusza, S., Gutman, D., Helba, B., Kalloo, A., Liopyris, K., Marchetti, M., Kittler, H., & Halpern, A. (2019). Skin Lesion Analysis Toward Melanoma Detection 2018: A Challenge Hosted by the International Skin Imaging Collaboration (ISIC). arXiv:1902.03368. https://arxiv.org/abs/1902.03368

Tschandl, P., Rosendahl, C., & Kittler, H. (2018). The HAM10000 dataset, a large collection of multi-source dermatoscopic images of common pigmented skin lesions. Scientific Data, 5, 180161. https://doi.org/10.1038/sdata.2018.161

Ronneberger, O., Fischer, P., & Brox, T. (2015). U-Net: Convolutional Networks for Biomedical Image Segmentation. arXiv:1505.04597. https://arxiv.org/abs/1505.04597

Oktay, O., Schlemper, J., Folgoc, L. L., Lee, M., Heinrich, M., Misawa, K., Mori, K., McDonagh, S., Hammerla, N. Y., Kainz, B., Glocker, B., & Rueckert, D. (2018). Attention U-Net: Learning Where to Look for the Pancreas. arXiv:1804.03999. https://arxiv.org/abs/1804.03999

